# 05 · Model comparison

Collects `results/*/metrics.json` and `test_predictions.npz` from all runs. Every model uses the
same data split, augmentation and test set, and each is selected on validation macro AUC.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch

from deeplense import CLASS_NAMES
from deeplense.utils import get_device, seed_everything

seed_everything(42)
DEVICE = get_device()
DATA_ROOT = ROOT / "data" / "lensing"
RESULTS = ROOT / "results"
print("device:", DEVICE)

In [ ]:
SMOKE = True   # compare the *_smoke runs, or the full runs when False
import json
runs = {}
for d in sorted(RESULTS.iterdir()):
    if (d / "metrics.json").exists() and d.name.endswith("_smoke") == SMOKE:
        runs[d.name] = (json.load(open(d / "metrics.json")), np.load(d / "test_predictions.npz"))

print(f"{'run':<28} {'params':>10} {'best ep':>8} {'test acc':>9} {'macro AUC':>10}  " +
      "  ".join(f"{n.split()[0][:8]:>8}" for n in CLASS_NAMES))
for name, (m, _) in runs.items():
    t = m["test"]
    print(f"{name:<28} {m['params']:>10,} {m['best_epoch']:>8} {t['accuracy']:>9.4f} {t['macro_auc']:>10.4f}  " +
          "  ".join(f"{v:>8.4f}" for v in t["auc_per_class"].values()))

In [ ]:
from sklearn.metrics import roc_curve
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for c, ax in enumerate(axes):
    for name, (m, p) in runs.items():
        y = (p["labels"] == c).astype(int)
        fpr, tpr, _ = roc_curve(y, p["probs"][:, c])
        ax.plot(fpr, tpr, lw=2, label=f"{name} ({m['test']['auc_per_class'][CLASS_NAMES[c]]:.4f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set(title=f"{CLASS_NAMES[c]} vs rest", xlabel="FPR", ylabel="TPR"); ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS / f"comparison_roc{'_smoke' if SMOKE else ''}.png", dpi=130, bbox_inches="tight"); plt.show()

## Discussion

*(Fill in after the full runs.)* Points to cover:
- CNN from scratch vs pretrained EfficientNet: how much ImageNet features help on
  non-natural, single-channel scientific images.
- PINN vs plain EfficientNet (same backbone): does the Poisson constraint improve AUC or sample
  efficiency, and what do ψ̂ / κ̂ / Ŝ look like? Use the λ ablation, not a single run.
- DeiT-Tiny vs CNNs: transformers lack a locality prior; is 27k images enough?
- Which class pair is hardest (see confusion matrices)?